# DCGAN on WikiArt (Optimized)

In [ ]:
!pip install scipy -q

In [ ]:
import os, random, copy
import numpy as np
import scipy.linalg
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.utils as vutils
import torchvision.transforms as T
from tqdm.auto import tqdm
from PIL import Image
import matplotlib.pyplot as plt

SEED        = 42
IMG_SIZE    = 64
NC          = 3
NZ          = 128
NGF         = 64
NDF         = 64
BATCH_SIZE  = 64
NUM_EPOCHS  = 500
LR_D        = 0.0002
LR_G        = 0.0002
BETA1       = 0.5
BETA2       = 0.999
EMA_DECAY   = 0.999
N_GEN       = 2
NOISE_STD   = 0.1
NOISE_ANNEAL = 200

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"PyTorch: {torch.__version__}")

## Data (Full GPU)

In [ ]:
DATA_ROOT = "/kaggle/input/wikiart"
STYLE = "Expressionism"

style_dir = os.path.join(DATA_ROOT, STYLE)
valid_ext = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
prep = T.Compose([T.Resize(IMG_SIZE, antialias=True), T.CenterCrop(IMG_SIZE),
                  T.ToTensor(), T.Normalize((0.5,)*3, (0.5,)*3)])

tensors = []
for fname in tqdm(sorted(os.listdir(style_dir)), desc=f"Loading {STYLE}"):
    if os.path.splitext(fname)[1].lower() not in valid_ext:
        continue
    try:
        tensors.append(prep(Image.open(os.path.join(style_dir, fname)).convert("RGB")))
    except Exception:
        pass

all_data = torch.stack(tensors).to(DEVICE, non_blocking=True)
del tensors
N_IMAGES = all_data.size(0)
N_BATCHES = N_IMAGES // BATCH_SIZE

print(f"Style: {STYLE}")
print(f"Images: {N_IMAGES:,} on {DEVICE}")
print(f"Batches/epoch: {N_BATCHES}")
print(f"GPU memory: {all_data.nbytes / 1e9:.2f} GB")

## Architecture

In [ ]:
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.main = nn.Sequential(
            nn.ConvTranspose2d(NZ, NGF*8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(NGF*8),
            nn.ReLU(True),

            nn.Upsample(scale_factor=2, mode='nearest'),
            nn.Conv2d(NGF*8, NGF*4, 3, 1, 1, bias=False),
            nn.BatchNorm2d(NGF*4),
            nn.ReLU(True),

            nn.Upsample(scale_factor=2, mode='nearest'),
            nn.Conv2d(NGF*4, NGF*2, 3, 1, 1, bias=False),
            nn.BatchNorm2d(NGF*2),
            nn.ReLU(True),

            nn.Upsample(scale_factor=2, mode='nearest'),
            nn.Conv2d(NGF*2, NGF, 3, 1, 1, bias=False),
            nn.BatchNorm2d(NGF),
            nn.ReLU(True),

            nn.Upsample(scale_factor=2, mode='nearest'),
            nn.Conv2d(NGF, NC, 3, 1, 1, bias=False),
            nn.Tanh(),
        )
    def forward(self, z): return self.main(z)

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.main = nn.Sequential(
            nn.Conv2d(NC, NDF, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(NDF, NDF*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(NDF*2),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(NDF*2, NDF*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(NDF*4),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(NDF*4, NDF*8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(NDF*8),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(NDF*8, 1, 4, 1, 0, bias=False),
        )
    def forward(self, x): return self.main(x).view(-1)

def weights_init(m):
    cn = m.__class__.__name__
    if 'Conv' in cn:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif 'BatchNorm' in cn:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

netG = Generator().to(DEVICE).apply(weights_init)
netD = Discriminator().to(DEVICE).apply(weights_init)

ema_G = copy.deepcopy(netG)
ema_G.eval()
for p in ema_G.parameters(): p.requires_grad_(False)

@torch.no_grad()
def update_ema(ema_m, m, decay):
    for ep, p in zip(ema_m.parameters(), m.parameters()):
        ep.data.mul_(decay).add_(p.data, alpha=1 - decay)

print(f"G: {sum(p.numel() for p in netG.parameters()):,} params")
print(f"D: {sum(p.numel() for p in netD.parameters()):,} params")

## Training

In [ ]:
optD = optim.Adam(netD.parameters(), lr=LR_D, betas=(BETA1, BETA2))
optG = optim.Adam(netG.parameters(), lr=LR_G, betas=(BETA1, BETA2))

fixed_noise = torch.randn(64, NZ, 1, 1, device=DEVICE)
G_losses, D_losses, img_snapshots = [], [], []
step = 0

for epoch in range(NUM_EPOCHS):
    netG.train(); netD.train()
    perm = torch.randperm(N_IMAGES, device=DEVICE)

    noise_std = NOISE_STD * max(0.0, 1.0 - epoch / NOISE_ANNEAL)
    add_noise = noise_std > 0

    d_buf = torch.zeros(N_BATCHES, device=DEVICE)
    g_buf = torch.zeros(N_BATCHES, device=DEVICE)

    for i in range(N_BATCHES):
        idx = perm[i * BATCH_SIZE : (i + 1) * BATCH_SIZE]
        real = all_data[idx]
        bsz = real.size(0)

        flip_mask = torch.rand(bsz, 1, 1, 1, device=DEVICE) > 0.5
        real = torch.where(flip_mask, real.flip(-1), real)

        if add_noise:
            real_in = real + noise_std * torch.randn_like(real)
        else:
            real_in = real

        netD.zero_grad(set_to_none=True)
        noise = torch.randn(bsz, NZ, 1, 1, device=DEVICE)
        with torch.no_grad():
            fake = netG(noise)
        fake_in = (fake + noise_std * torch.randn_like(fake)) if add_noise else fake

        errD = F.relu(1.0 - netD(real_in)).mean() + F.relu(1.0 + netD(fake_in)).mean()
        errD.backward()
        optD.step()

        g_acc = torch.zeros(1, device=DEVICE)
        for _ in range(N_GEN):
            netG.zero_grad(set_to_none=True)
            noise = torch.randn(bsz, NZ, 1, 1, device=DEVICE)
            errG = -netD(netG(noise)).mean()
            errG.backward()
            optG.step()
            g_acc += errG.detach()

        ema_d = min(EMA_DECAY, (1 + step) / (10 + step))
        update_ema(ema_G, netG, ema_d)

        d_buf[i] = errD.detach()
        g_buf[i] = g_acc / N_GEN
        step += 1

    d_cpu = d_buf.cpu().tolist()
    g_cpu = g_buf.cpu().tolist()
    D_losses.extend(d_cpu)
    G_losses.extend(g_cpu)
    avg_d = sum(d_cpu) / N_BATCHES
    avg_g = sum(g_cpu) / N_BATCHES
    print(f"Epoch {epoch+1:03d}/{NUM_EPOCHS} | D: {avg_d:.3f} | G: {avg_g:.3f} | noise: {noise_std:.3f} | {step:,} steps")

    with torch.no_grad():
        img_snapshots.append(vutils.make_grid(ema_G(fixed_noise).cpu(), padding=2, normalize=True))

    if (epoch + 1) % 50 == 0:
        torch.save(ema_G.state_dict(), f"/kaggle/working/ema_G_ep{epoch+1}.pt")

print(f"\nDone. {step:,} total steps.")

## Loss Curves

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(G_losses, alpha=0.6, label="Generator")
plt.plot(D_losses, alpha=0.6, label="Discriminator")
plt.xlabel("Iteration"); plt.ylabel("Loss")
plt.title("Training Loss")
plt.legend(); plt.tight_layout()
plt.savefig("/kaggle/working/loss_curves.png", dpi=150)
plt.show()

## FID & IS

In [ ]:
from torchvision.models import inception_v3, Inception_V3_Weights

torch.cuda.empty_cache()
NUM_EVAL = min(5000, N_IMAGES)

inception = inception_v3(weights=Inception_V3_Weights.DEFAULT, transform_input=False).to(DEVICE).eval()
_fc_w = inception.fc.weight.data.clone()
_fc_b = inception.fc.bias.data.clone()
inception.fc = nn.Identity()

@torch.no_grad()
def _extract(img_iter, n):
    feats, probs = [], []
    count = 0
    for batch in img_iter:
        batch = F.interpolate(batch, 299, mode='bilinear', align_corners=False)
        f = inception(batch)
        logits = F.linear(f, _fc_w, _fc_b)
        feats.append(f.cpu())
        probs.append(torch.softmax(logits, dim=1).cpu())
        count += batch.size(0)
        if count >= n:
            break
    return torch.cat(feats)[:n].numpy(), torch.cat(probs)[:n].numpy()

def calc_fid(f1, f2):
    mu1, mu2 = f1.mean(0), f2.mean(0)
    s1 = np.cov(f1, rowvar=False) + np.eye(f1.shape[1]) * 1e-6
    s2 = np.cov(f2, rowvar=False) + np.eye(f2.shape[1]) * 1e-6
    diff = mu1 - mu2
    covmean, _ = scipy.linalg.sqrtm(s1 @ s2, disp=False)
    if np.iscomplexobj(covmean): covmean = covmean.real
    return float(diff @ diff + np.trace(s1 + s2 - 2 * covmean))

def calc_is(probs, splits=10):
    chunk = len(probs) // splits
    scores = []
    for k in range(splits):
        p = probs[k * chunk:(k + 1) * chunk]
        q = p.mean(0, keepdims=True)
        kl = (p * (np.log(p + 1e-10) - np.log(q + 1e-10))).sum(1)
        scores.append(np.exp(kl.mean()))
    return float(np.mean(scores)), float(np.std(scores))

def real_iter():
    for i in range(0, N_IMAGES, 64):
        yield (all_data[i:i+64] + 1) / 2

def fake_iter():
    rem = NUM_EVAL
    while rem > 0:
        bsz = min(64, rem)
        yield (ema_G(torch.randn(bsz, NZ, 1, 1, device=DEVICE)) + 1) / 2
        rem -= bsz

print("Extracting real features...")
feats_real, _ = _extract(real_iter(), NUM_EVAL)
print("Extracting fake features...")
feats_fake, probs_fake = _extract(fake_iter(), NUM_EVAL)

fid_score = calc_fid(feats_real, feats_fake)
is_score, is_std = calc_is(probs_fake)
print(f"FID: {fid_score:.2f}")
print(f"IS : {is_score:.2f} ± {is_std:.2f}")

## Generated Images

In [ ]:
real_batch = all_data[:64]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, grid, title in zip(
    axes,
    [
        vutils.make_grid(real_batch.cpu(), padding=2, normalize=True),
        img_snapshots[-1],
    ],
    [f"Real ({STYLE})", "Generated (EMA)"],
):
    ax.imshow(np.transpose(grid.numpy(), (1, 2, 0)))
    ax.set_title(title, fontsize=14)
    ax.axis("off")

plt.tight_layout()
plt.savefig("/kaggle/working/real_vs_generated.png", dpi=150)
plt.show()

In [ ]:
with torch.no_grad():
    samples = ema_G(torch.randn(16, NZ, 1, 1, device=DEVICE)).cpu()

fig, axes = plt.subplots(4, 4, figsize=(10, 10))
for ax, img in zip(axes.flat, samples):
    ax.imshow(np.clip(img.permute(1, 2, 0).numpy() * 0.5 + 0.5, 0, 1))
    ax.axis("off")
plt.suptitle(f"Generated ({STYLE})  |  FID: {fid_score:.1f}  IS: {is_score:.2f}", fontsize=13)
plt.tight_layout()
plt.savefig("/kaggle/working/generated_examples.png", dpi=150)
plt.show()

## Training Progression

In [ ]:
show_epochs = list(range(0, len(img_snapshots), max(1, len(img_snapshots) // 10)))
if (len(img_snapshots) - 1) not in show_epochs:
    show_epochs.append(len(img_snapshots) - 1)

cols = min(len(show_epochs), 5)
rows = (len(show_epochs) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 4))
axes = np.array(axes).flat
for ax, idx in zip(axes, show_epochs):
    ax.imshow(np.transpose(img_snapshots[idx], (1, 2, 0)))
    ax.set_title(f"Epoch {idx + 1}", fontsize=9)
    ax.axis("off")
for ax in list(axes)[len(show_epochs):]:
    ax.axis("off")
plt.suptitle(f"{STYLE} — Training Progression", fontsize=13)
plt.tight_layout()
plt.savefig("/kaggle/working/training_progression.png", dpi=120)
plt.show()